# 🔥 Bot de Ofertas — Mercado Livre → WhatsApp
Busca ofertas com desconto no Mercado Livre e posta automaticamente no seu grupo do WhatsApp.

**Execute as células na ordem de cima para baixo.**

In [ ]:
# ── CÉLULA 1 — Instalar dependências ──────────────────────────
!pip install requests python-dotenv -q

In [ ]:
# ── CÉLULA 2 — Configurações (preencha aqui) ──────────────────

# Mercado Livre
ML_APP_ID        = "1021146102101711"
ML_SECRET_KEY    = "kLeXe47oPqX69bcNjuNM7nfQbZpDIohN"
ML_SITE_ID       = "MLB"  # MLB = Brasil

# Filtros
KEYWORDS             = ["iphone", "samsung", "notebook", "playstation"]
CATEGORIES           = []          # ex: ["MLB1051"] — deixe [] para todas
MIN_DISCOUNT_PERCENT = 15          # desconto mínimo em %
MAX_PRICE            = 0           # preço máximo em R$ (0 = sem limite)
MAX_OFFERS_PER_RUN   = 5           # máximo de mensagens por rodada

# Green API (WhatsApp)
GREEN_API_ID_INSTANCE = "7107621368"
GREEN_API_TOKEN       = "2a141bed4be64203ab153a30baccd1afee8b9f2e16924eb7aa"
WHATSAPP_GROUP_ID     = "120363407602900721@g.us"

# Intervalo entre buscas (minutos)
SCHEDULE_INTERVAL_MINUTES = 30

print("✅ Configurações salvas!")

In [ ]:
# ── CÉLULA 3 — Código do bot ──────────────────────────────────
import requests
import json
import time
import logging
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

ML_API_BASE = "https://api.mercadolibre.com"
SEARCH_URL  = f"{ML_API_BASE}/sites/{ML_SITE_ID}/search"
TOKEN_URL   = f"{ML_API_BASE}/oauth/token"
SENT_FILE   = Path("sent_offers.json")
TTL_SECONDS = 7 * 24 * 60 * 60

_access_token = None


@dataclass
class Offer:
    id: str
    title: str
    price: float
    original_price: Optional[float]
    discount_percent: float
    url: str
    seller: str
    available_quantity: int

    def format_message(self) -> str:
        if self.original_price and self.discount_percent > 0:
            price_line = (
                f"De: R$ {self.original_price:,.2f}\n"
                f"Por: *R$ {self.price:,.2f}*\n"
                f"📉 *{self.discount_percent:.0f}% de desconto!*"
            )
        else:
            price_line = f"💰 *R$ {self.price:,.2f}*"
        return (
            f"🔥 *OFERTA ENCONTRADA!*\n\n"
            f"📦 {self.title}\n\n"
            f"{price_line}\n\n"
            f"🏪 Vendedor: {self.seller}\n"
            f"📦 Estoque: {self.available_quantity} un\n\n"
            f"🔗 {self.url}"
        )


# ── Mercado Livre ──────────────────────────────────────────────

def get_ml_token() -> Optional[str]:
    global _access_token
    if _access_token:
        return _access_token
    try:
        resp = requests.post(TOKEN_URL, data={
            "grant_type": "client_credentials",
            "client_id": ML_APP_ID,
            "client_secret": ML_SECRET_KEY,
        }, timeout=15)
        resp.raise_for_status()
        _access_token = resp.json().get("access_token")
        logger.info("Token ML obtido com sucesso")
        return _access_token
    except Exception as e:
        logger.error("Erro ao obter token ML: %s", e)
        return None


def fetch_offers(keyword="", category="") -> list:
    headers = {"Accept": "application/json"}
    token = get_ml_token()
    if token:
        headers["Authorization"] = f"Bearer {token}"

    params = {"limit": 50, "sort": "best_match"}
    if keyword:  params["q"] = keyword
    if category: params["category"] = category

    try:
        resp = requests.get(SEARCH_URL, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
        items = resp.json().get("results", [])
    except Exception as e:
        logger.error("Erro ao buscar ML: %s", e)
        return []

    offers = []
    for item in items:
        try:
            price = float(item["price"])
            orig  = float(item["original_price"]) if item.get("original_price") else None
            disc  = round((1 - price / orig) * 100, 1) if orig else 0.0

            if disc < MIN_DISCOUNT_PERCENT: continue
            if MAX_PRICE and price > MAX_PRICE: continue

            offers.append(Offer(
                id=item["id"],
                title=item["title"],
                price=price,
                original_price=orig,
                discount_percent=disc,
                url=item.get("permalink", ""),
                seller=item.get("seller", {}).get("nickname", "?"),
                available_quantity=item.get("available_quantity", 0),
            ))
        except Exception:
            continue

    logger.info("Busca '%s': %d aprovadas de %d", keyword or category, len(offers), len(items))
    return offers


def fetch_all_offers() -> list:
    seen, all_offers = set(), []
    combos = []
    if KEYWORDS and CATEGORIES:
        combos = [(k, c) for k in KEYWORDS for c in CATEGORIES]
    elif KEYWORDS:
        combos = [(k, "") for k in KEYWORDS]
    elif CATEGORIES:
        combos = [("", c) for c in CATEGORIES]
    else:
        combos = [("", "")]

    for kw, cat in combos:
        for o in fetch_offers(kw, cat):
            if o.id not in seen:
                seen.add(o.id)
                all_offers.append(o)

    all_offers.sort(key=lambda o: o.discount_percent, reverse=True)
    return all_offers


# ── WhatsApp ───────────────────────────────────────────────────

def send_whatsapp(message: str) -> bool:
    url = (
        f"https://api.green-api.com/waInstance{GREEN_API_ID_INSTANCE}"
        f"/sendMessage/{GREEN_API_TOKEN}"
    )
    try:
        resp = requests.post(url, json={"chatId": WHATSAPP_GROUP_ID, "message": message}, timeout=20)
        resp.raise_for_status()
        logger.info("✅ Mensagem enviada!")
        return True
    except Exception as e:
        logger.error("Erro ao enviar WhatsApp: %s", e)
        return False


# ── Controle de duplicatas ─────────────────────────────────────

def load_sent() -> dict:
    if not SENT_FILE.exists(): return {}
    try:
        return json.loads(SENT_FILE.read_text())
    except Exception:
        return {}

def save_sent(data: dict):
    cutoff = int(time.time()) - TTL_SECONDS
    data = {k: v for k, v in data.items() if v >= cutoff}
    SENT_FILE.write_text(json.dumps(data, indent=2))


# ── Execução ───────────────────────────────────────────────────

def run_once():
    logger.info("🔍 Buscando ofertas...")
    all_offers = fetch_all_offers()
    sent_data  = load_sent()
    new_offers = [o for o in all_offers if o.id not in sent_data]
    to_send    = new_offers[:MAX_OFFERS_PER_RUN]

    logger.info("%d novas de %d encontradas — enviando %d",
                len(new_offers), len(all_offers), len(to_send))

    for offer in to_send:
        if send_whatsapp(offer.format_message()):
            sent_data[offer.id] = int(time.time())
            time.sleep(2)  # pequena pausa entre mensagens

    save_sent(sent_data)


print("✅ Código carregado com sucesso!")

In [ ]:
# ── CÉLULA 4 — Testar uma vez ─────────────────────────────────
# Execute esta célula para fazer uma busca agora e ver se funciona
run_once()

In [ ]:
# ── CÉLULA 5 — Rodar em loop contínuo ────────────────────────
# Execute esta célula para deixar o bot rodando automaticamente.
# ⚠️ O Colab gratuito desconecta após ~90min sem interação.
# Dica: clique na aba do Colab de vez em quando para manter ativo.

import datetime

print(f"🤖 Bot iniciado! Buscando a cada {SCHEDULE_INTERVAL_MINUTES} minutos...")
print("Para parar: clique em Runtime → Interrupt execution\n")

while True:
    try:
        run_once()
    except Exception as e:
        logger.error("Erro inesperado: %s", e)

    proximo = datetime.datetime.now() + datetime.timedelta(minutes=SCHEDULE_INTERVAL_MINUTES)
    print(f"⏳ Próxima busca às {proximo.strftime('%H:%M')}...")
    time.sleep(SCHEDULE_INTERVAL_MINUTES * 60)